#### Creat a target table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_target (
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.default.orders_target
VALUES (101, 'Arun', 'Pending', 500),
       (102, 'Priya', 'Completed', 800),
       (103, 'John', 'Pending',300)

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

#### INSERT USING MERGE

#### Create Source Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_source (
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
USING DELTA;

#### Add one new order

In [0]:
%sql
INSERT INTO workspace.default.orders_source
VALUES (104,'David','Pending',600)

#### View the incoming order

In [0]:
%sql
SELECT * FROM workspace.default.orders_source;

MERGE with INSERT only


In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN NOT MATCHED THEN 
INSERT (
    target.order_id,
    target.customer_name,
    target.order_status,
    target.amount
)
VALUES (
    source.order_id,
    source.customer_name,
    source.order_status,
    source.amount
)

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source
VALUES (101,'Arun','Completed',500);

In [0]:
%sql
SELECT * FROM workspace.default.orders_source;

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

#### Update Using MERGE


In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN 
UPDATE SET 
    target.customer_name = source.customer_name,
    target.order_status = source.order_status,
    target.amount = source.amount 
    

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.orders_target;

#### INSERT AND UPDATE TOGETHER

##### Prepare two incoming records

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(102,'Priya','Refunded',800),
(105,'Sarah','Shipped',1000),
(106,'Ravi','Cancelled',1500)

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id


WHEN MATCHED THEN
UPDATE
SET
    target.customer_name = source.customer_name,
    target.order_status = source.order_status,
    target.amount = source.amount

WHEN NOT MATCHED THEN
INSERT (
    target.order_id,
    target.customer_name,
    target.order_status,
    target.amount
)
VALUES (
    source.order_id,
    source.customer_name,
    source.order_status,
    source.amount
)

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id != source.order_id

WHEN MATCHED THEN
DELETE;
    
SELECT * FROM workspace.default.orders_target;

In [0]:
%sql
DESCRIBE HISTORY workspace.default.orders_target;

In [0]:
%sql
RESTORE TABLE workspace.default.orders_target TO VERSION AS OF 8;

In [0]:
%sql
SELECT * FROM workspace.default.orders_target;

In [0]:
%sql
-- DELETE using MERGE with a NULL order_id record
-- A NULL id never matches with a plain equality join (=),
-- so we use the null-safe operator <=> to match NULL keys

TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(NULL, 'Kiran', 'Pending', 700);

MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id <=> source.order_id

WHEN MATCHED THEN
DELETE;

SELECT * FROM workspace.default.orders_target;

#### What happens if multiple source records match the same target records?


In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(101,'Arun','Shipped',500),
(101,'Arun','Delivered',500)

#### View Duplicate Records

In [0]:
%sql
SELECT * FROM workspace.default.orders_source;

Execute the MERGE AND Observer Error

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET 
    target.order_status = source.order_status,
    target.amount = source.amount;

Deduplicate

In [0]:
%sql

-- Deduplicate the source: keep only the latest row per order_id
CREATE OR REPLACE VIEW workspace.default.orders_source_dedup AS
SELECT order_id, customer_name, order_status, amount
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY amount DESC) AS rn
    FROM workspace.default.orders_source
)
WHERE rn = 1;

-- Re-run the MERGE using the deduplicated source
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source_dedup AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET 
    target.customer_name = source.customer_name,
    target.order_status = source.order_status,
    target.amount = source.amount;

SELECT * FROM workspace.default.orders_target;
